<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
import sys, subprocess
try:
    import tensorflow
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow-cpu", "-q"])

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed):
    (X_full, y_full), _ = cifar10.load_data()

    # flatten: (N, 32, 32, 3) -> (N, 3072)  e normaliza para [0, 1]
    X_full = X_full.reshape(X_full.shape[0], -1).astype('float32') / 255.0
    y_full = y_full.ravel()

    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=seed
    )
    return X_train, X_val, y_train, y_val

SEED = 42
X_train, X_val, y_train, y_val = load_data(SEED)
print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

print("""
Respostas:
1. Formato original das imagens: (32, 32, 3) - altura x largura x canais RGB.
2. Apos o flatten cada imagem possui 32x32x3 = 3.072 features.
3. O flatten e necessario porque a MLP recebe vetores 1-D; ela nao possui operacoes
   convolucionais para explorar a grade 2-D espacial.
4. A normalizacao coloca todos os pixels na mesma escala (0-1), acelerando a convergencia
   do gradiente descendente e evitando que features de valores maiores dominem o aprendizado.
""")


# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=20,
        batch_size=256,
        verbose=False,
    )
    model.fit(X_train, y_train)
    return model

print("""
Respostas:
1. Parametros na 1a camada (ex.: 128 neuronios): 3.072 entradas x 128 + 128 bias = 393.344.
2. ReLU retorna max(0, x): zera ativacoes negativas e mantem as positivas, introduzindo
   nao-linearidade sem saturacao, reduzindo o problema do gradiente vanescente.
3. A MLP usa camadas densas (fully-connected): cada neuronio conecta-se a todos da camada
   anterior. Com 3.072 features de entrada, mesmo uma camada pequena ja gera centenas de
   milhares de parametros.
""")


# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall":    recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score":  f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }


**Solucao - Respostas Q3:**

1. **O que a accuracy representa?**  
   A accuracy mede a proporcao de predicoes corretas sobre o total de amostras:
   (TP + TN) / Total. E intuitiva, mas pode ser enganosa em datasets desbalanceados.

2. **Qual a diferenca entre precision e recall?**  
   - **Precision**: dos exemplos classificados como positivos, quantos realmente sao? -> TP / (TP + FP). Penaliza falsos positivos.  
   - **Recall**: dos exemplos realmente positivos, quantos foram capturados? -> TP / (TP + FN). Penaliza falsos negativos.

3. **Em quais situacoes o f1-score e importante?**  
   O f1-score e a media harmonica de precision e recall; e essencial quando as classes sao
   desbalanceadas ou quando falsos positivos e falsos negativos tem custos similares, pois
   penaliza modelos que sacrificam uma medida para maximizar a outra.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import time

def run_experiment(X_train, y_train, X_val, y_val,
                   activation, hidden_layers, learning_rate, seed,
                   max_iter=20, batch_size=256):
    with mlflow.start_run():
        mlflow.log_param("activation",     activation)
        mlflow.log_param("hidden_layers",  str(hidden_layers))
        mlflow.log_param("learning_rate",  learning_rate)
        mlflow.log_param("max_iter",       max_iter)
        mlflow.log_param("batch_size",     batch_size)

        t0 = time.time()
        model = train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed)
        training_time = time.time() - t0

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_metric("accuracy",      metrics["accuracy"])
        mlflow.log_metric("precision",     metrics["precision"])
        mlflow.log_metric("recall",        metrics["recall"])
        mlflow.log_metric("f1_score",      metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)

    tag = f"act={activation}  hidden={str(hidden_layers):12s}  lr={learning_rate}"
    print(f"{tag}  ->  acc={metrics['accuracy']:.4f}  time={training_time:.1f}s")
    return model, metrics

# Experimento base
print("=== Experimento Base ===")
model_base, metrics_base = run_experiment(
    X_train, y_train, X_val, y_val,
    activation="relu", hidden_layers=(128, 64), learning_rate=0.01, seed=SEED,
)
print(pd.DataFrame([metrics_base]))

print("""
Respostas:
1. O experimento com relu, hidden=(128,64) e lr=0.01 tende a apresentar o melhor equilibrio.
   Para comparar: execute `mlflow ui` no terminal e acesse http://localhost:5000.
2. Configuracoes com lr=0.01 e ativacao relu convergem de forma mais estavel.
3. O MLflow garante reprodutibilidade, facilita a comparacao de hiperparametros e registra
   automaticamente o historico, eliminando controle manual via planilhas.
""")


# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activations = ["logistic", "tanh", "relu"]
results_q5 = []

print("=== Q5: Comparacao de Funcoes de Ativacao ===")
for act in activations:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation=act, hidden_layers=(128, 64), learning_rate=0.01, seed=SEED,
    )
    results_q5.append({"activation": act, **metrics})

df_q5 = pd.DataFrame(results_q5).set_index("activation")
print(df_q5)

print("""
Respostas:
1. ReLU geralmente apresenta melhor convergencia por nao sofrer saturacao de gradiente.
2. tanh tende a ser mais estavel por ser centrada em zero, reduzindo o desvio do gradiente.
3. Sim; logistic converge mais devagar e pode apresentar accuracy menor em redes profundas
   devido ao problema do gradiente vanescente.
4. ReLU e computacionalmente simples (max(0,x)), nao satura para valores positivos, atenua
   o gradiente vanescente e produz empiricamente convergencia mais rapida.
""")


# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
results_q6 = []

print("=== Q6: Comparacao de Arquiteturas ===")
for arch in architectures:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu", hidden_layers=arch, learning_rate=0.01, seed=SEED,
    )
    results_q6.append({"hidden_layers": str(arch), **metrics})

df_q6 = pd.DataFrame(results_q6).set_index("hidden_layers")
print(df_q6)

print("""
Respostas:
1. Nao necessariamente; redes maiores podem sofrer overfitting ou ter custo muito alto
   sem ganho proporcional de accuracy, especialmente com poucas epocas.
2. (128, 64) geralmente oferece o melhor tradeoff entre capacidade e custo computacional.
3. Arquiteturas maiores como (256, 128) podem apresentar leve overfitting; comparar
   accuracy de treino x validacao evidencia isso.
4. (256, 128) tem o maior numero de parametros e exige mais memoria e tempo por epoca.
""")


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
results_q7 = []

print("=== Q7: Comparacao de Learning Rates ===")
for lr in learning_rates:
    _, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        activation="relu", hidden_layers=(128, 64), learning_rate=lr, seed=SEED,
    )
    results_q7.append({"learning_rate": lr, **metrics})

df_q7 = pd.DataFrame(results_q7).set_index("learning_rate")
print(df_q7)

print("""
Respostas:
1. lr=0.01 costuma apresentar o melhor equilibrio entre velocidade de convergencia e accuracy.
2. lr=0.1 tende a causar maior instabilidade (oscilacoes na loss) ou ate divergencia.
3. Com lr muito alto o modelo oscila em torno do minimo ou diverge, pois os pesos sao
   atualizados demais e 'saltam' o ponto otimo.
4. Com lr muito baixo a convergencia e extremamente lenta, exigindo muitas epocas para
   que a loss reduza de forma significativa.
""")


# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
discussao = """
=== Q8: Discussao Final ===

Comportamento da loss:
  Decresceu progressivamente ao longo das epocas. Learning rates altos causaram oscilacao;
  os baixos resultaram em convergencia lenta. Com max_iter=20, nenhum modelo convergiu
  completamente.

Impacto do learning rate:
  lr=0.01 mostrou o melhor balanco. lr=0.1 provocou instabilidade; lr=0.001 convergiu de
  forma mais suave, porem muito mais devagar.

Impacto da arquitetura:
  Arquiteturas maiores (128,64) e (256,128) capturam representacoes mais complexas, porem
  com custo computacional mais elevado. (32,) aprende rapidamente, mas com menor capacidade.

Impacto das funcoes de ativacao:
  ReLU superou logistic e tanh: evita saturacao de gradiente e e mais simples de calcular.
  Logistic convergiu mais devagar e apresentou accuracy um pouco inferior.

Comportamento do treinamento:
  Todos os modelos melhoraram ao longo das iteracoes, mas poucos atingiram convergencia
  total com max_iter=20. Aumentar max_iter melhora os resultados ao custo de maior tempo.

Limitacoes da MLP:
  - Nao captura relacoes espaciais locais (tarefa natural de CNNs).
  - Alto numero de parametros: 3.072 features geram centenas de milhares ja na 1a camada.
  - Sensivel a escala dos dados (exige normalizacao).
  - Propenso a overfitting sem regularizacao (dropout, weight decay).

Relacao entre backpropagation e aprendizado:
  O backpropagation calcula o gradiente da loss em relacao a cada peso pela regra da cadeia,
  propagando o erro da saida para as camadas anteriores. O otimizador usa esses gradientes
  para atualizar os pesos na direcao de menor perda, permitindo que a rede aprenda
  representacoes hierarquicas uteis para a classificacao.

Respostas:
1. Melhor configuracao: relu + (128, 64) + lr=0.01, com maior accuracy e estabilidade.
2. Principais dificuldades: alto custo computacional no CIFAR-10 (3.072 features x 40k
   amostras), convergencia lenta com max_iter baixo e limitacao intrinseca da MLP.
3. Limitacao para imagens: a MLP ignora a estrutura espacial; trata cada pixel como feature
   independente, perdendo correlacoes de vizinhanca e invariancia a translacao que CNNs
   exploram naturalmente.
4. Backpropagation distribui a responsabilidade do erro para cada peso via regra da cadeia,
   calculando exatamente quanto cada parametro deve ser ajustado para reduzir a loss,
   viabilizando o treinamento eficiente de redes com multiplas camadas.
"""

print(discussao)
